# 共起する語の変化から意味の変化を読み取る (PPMI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_lexical_semantics/semantic_change_ppmi.ipynb)

NLP クックブック 17-08 ／ 語彙知識と意味の変化

ある単語と一緒に使われる語が、時代によってどう変わったかを直接見たい。ニューラルネットを使わず、共起の統計だけで調べる。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import math, random, re
from collections import Counter, defaultdict
from datasets import load_dataset
from fugashi import Tagger

ds = load_dataset("globis-university/aozorabunko-clean", split="train")
tagger = Tagger()
random.seed(0)

def era(meta):
    year = (meta.get("生年月日") or "")[:4]
    if meta["文字遣い種別"] != "新字新仮名" or not year.isdigit():
        return None
    return "旧" if 1850 <= int(year) < 1880 else "新" if 1895 <= int(year) < 1930 else None

texts = {"旧": [], "新": []}
for ex in ds:
    if era(ex["meta"]):
        texts[era(ex["meta"])].append(ex["text"])

TARGETS = ["原子", "同志", "活動", "戦争", "自由", "言葉", "必要", "機械", "放送", "電話"]
CONTENT = {"名詞", "動詞", "形容詞", "形状詞"}

def cooccurrence(works, window=5, max_chars=20_000_000):
    """対象語の前後 window 語に出てくる内容語を数える"""
    pair, word, n = defaultdict(Counter), Counter(), 0
    for text in random.sample(works, len(works)):
        for s in re.split(r"(?<=[。！？])|\n", text[:30000]):
            words = [w.surface if w.feature.pos2 == "固有名詞" else (w.feature.lemma or w.surface).split("-")[0]     # 活用語は原形にまとめる
                     for w in tagger(s) if w.feature.pos1 in CONTENT]
            word.update(words)
            for i, w in enumerate(words):
                if w in TARGETS:
                    pair[w].update(words[max(0, i - window):i] + words[i + 1:i + 1 + window])
        n += min(len(text), 30000)
        if n > max_chars:
            break
    return pair, word

stats = {e: cooccurrence(t) for e, t in texts.items()}
print({e: (sum(word.values()), {t: sum(pair[t].values()) for t in TARGETS[:4]}) for e, (pair, word) in stats.items()})

{'旧': (5328205, {'原子': 823, '同志': 1426, '活動': 4797, '戦争': 7280}), '新': (5038620, {'原子': 3641, '同志': 3889, '活動': 8601, '戦争': 17763})}


In [3]:
def ppmi(e, target, min_count=5):
    """PPMI(対象語, 文脈語) = max(0, log2 [P(対象語, 文脈語) / (P(対象語) P(文脈語))])"""
    pair, word = stats[e]
    total, n_target = sum(word.values()), sum(pair[target].values())
    return {c: max(0.0, math.log2((n / n_target) / (word[c] / total))) for c, n in pair[target].items() if n >= min_count}

def cosine(a, b):
    dot = sum(v * b.get(k, 0.0) for k, v in a.items())
    return dot / (math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values())))

for t in TARGETS:
    old, new = ppmi("旧", t), ppmi("新", t)
    print(f"{t}: 類似度 {cosine(old, new):.2f}")

原子: 類似度 0.29
同志: 類似度 0.21
活動: 類似度 0.27
戦争: 類似度 0.30
自由: 類似度 0.42
言葉: 類似度 0.38
必要: 類似度 0.37
機械: 類似度 0.21
放送: 類似度 0.34
電話: 類似度 0.42


In [4]:
# どの共起語が増えて、どの共起語が減ったか
def diff(target, k=10):
    old, new = ppmi("旧", target), ppmi("新", target)
    delta = {c: new.get(c, 0.0) - old.get(c, 0.0) for c in set(old) | set(new)}
    ranked = sorted(delta, key=delta.get)
    return ranked[:k], ranked[::-1][:k]

for t in ["原子", "同志", "活動", "放送"]:
    lost, gained = diff(t)
    print(f"{t}\n   旧い時代に特徴的: {' '.join(lost)}\n   新しい時代に特徴的: {' '.join(gained)}")

原子
   旧い時代に特徴的: 分子 成り立つ 量 実在 区別 説明 運動 関する 考え 電子
   新しい時代に特徴的: 有鈎 変成 ヘリウム 核 ロリウム 爆弾 水素 主流 配列 兵器
同志
   旧い時代に特徴的: 社 話 心 会 若い 者 仲間 物 人間 知る
   新しい時代に特徴的: 古味 糾合 多喜二 重根 同志 小林 恋人 安 喧嘩 味方
活動
   旧い時代に特徴的: 弁士 偉人 開発 フィルム 勢力 法則 刺激 意志 成功 世紀
   新しい時代に特徴的: サークル 啓蒙 文筆 旺盛 創作 クリティシズム 精力 開始 キュリー 活発
放送
   旧い時代に特徴的: ラジオ 放送 東京 二 所 月 局 有る 来る 聞く
   新しい時代に特徴的: 中継 ＪＯＡＫ 番組 録音 聴取 臨時 協会 テレビ 受信 プログラム


- PPMI のベクトルは、各次元が「どの語と共起するか」をそのまま表す。時代が違っても次元の意味が同じなので、位置合わせなしで比べられ、どの共起語が変化に効いたかも読める。
- word2vec のような密なベクトルより、変化の中身を説明しやすい。「原子」と「爆弾」の共起が増えた、のように具体的な根拠を示せる。
- 頻度の低い語どうしの PMI は不安定に大きくなる。共起回数の下限を設ける、平滑化する、といった対処が要る。
- 対象語の頻度が時代で大きく違うと、共起語の種類の数が変わり、それだけで類似度が下がる。頻度の影響を差し引いて解釈する。